# MotoGP Podium Lockouts - Advanced Modeling and Analysis

**Dataset Focus**: `same_nation_podium_lockouts.csv`  
**CRISP-DM Phase**: 4 - Modeling  
**Purpose**: Advanced analysis of national dominance patterns and podium lockout phenomena

## Business Focus
- National dominance modeling
- Lockout frequency prediction
- Circuit-specific dominance patterns

In [ ]:
# Setup and data loading
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Load prepared podium lockouts data
data_path = Path("../../data/interim/")
df = pd.read_csv(data_path / "podium_lockouts_prepared.csv")

print(f"Podium lockouts data loaded: {df.shape}")
print(f"Date range: {df['season'].min()} - {df['season'].max()}")
print(f"Unique nations: {df['nation_clean'].nunique()}")
print(f"Total lockout events: {len(df)}")
df.head()

## National Dominance Analysis

In [ ]:
print("=== NATIONAL DOMINANCE ANALYSIS ===")

# Nation dominance metrics
nation_metrics = df.groupby('nation_clean').agg({
    'season': ['count', 'min', 'max', 'nunique'],
    'track_clean': 'nunique',
    'class_clean': 'nunique'
})

nation_metrics.columns = ['total_lockouts', 'first_lockout', 'last_lockout', 'seasons_active', 'tracks_dominated', 'classes_dominated']
nation_metrics['lockout_span'] = nation_metrics['last_lockout'] - nation_metrics['first_lockout']
nation_metrics['lockouts_per_season'] = nation_metrics['total_lockouts'] / nation_metrics['seasons_active']
nation_metrics['dominance_index'] = nation_metrics['total_lockouts'] * nation_metrics['tracks_dominated'] * nation_metrics['classes_dominated']

# Sort by total lockouts
nation_metrics = nation_metrics.sort_values('total_lockouts', ascending=False)

print("Top 10 nations by lockout dominance:")
print(nation_metrics.head(10).round(2))

# Dominance categories
def categorize_dominance(row):
    if row['total_lockouts'] >= 50:
        return 'Extreme Dominance'
    elif row['total_lockouts'] >= 20:
        return 'High Dominance'
    elif row['total_lockouts'] >= 10:
        return 'Moderate Dominance'
    elif row['total_lockouts'] >= 5:
        return 'Limited Dominance'
    else:
        return 'Rare Lockouts'

nation_metrics['dominance_category'] = nation_metrics.apply(categorize_dominance, axis=1)

print(f"\nDominance categories:")
dominance_dist = nation_metrics['dominance_category'].value_counts()
for category, count in dominance_dist.items():
    avg_lockouts = nation_metrics[nation_metrics['dominance_category'] == category]['total_lockouts'].mean()
    print(f"{category}: {count} nations (avg: {avg_lockouts:.1f} lockouts)")

# Visualization
plt.figure(figsize=(14, 8))
top_15 = nation_metrics.head(15)
plt.barh(range(len(top_15)), top_15['total_lockouts'], color='crimson', alpha=0.7)
plt.yticks(range(len(top_15)), top_15.index)
plt.xlabel('Total Podium Lockouts')
plt.title('Top 15 Nations by Podium Lockout Frequency')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Temporal and Circuit Analysis

In [ ]:
print("=== TEMPORAL AND CIRCUIT LOCKOUT ANALYSIS ===")

# Era-based lockout frequency
era_lockouts = df.groupby(['era', 'nation_clean']).size().reset_index(name='lockouts')
era_summary = era_lockouts.groupby('era').agg({
    'lockouts': ['sum', 'mean', 'std'],
    'nation_clean': 'nunique'
})
era_summary.columns = ['total_lockouts', 'avg_per_nation', 'std_lockouts', 'nations_involved']

print("Lockout patterns by era:")
print(era_summary.round(2))

# Top nations by era
print(f"\nDominant nations by era:")
for era in df['era'].unique():
    era_data = era_lockouts[era_lockouts['era'] == era]
    top_nation = era_data.nlargest(1, 'lockouts')
    if len(top_nation) > 0:
        nation = top_nation.iloc[0]['nation_clean']
        lockouts = top_nation.iloc[0]['lockouts']
        era_total = era_data['lockouts'].sum()
        percentage = lockouts / era_total * 100 if era_total > 0 else 0
        print(f"{era}: {nation} ({lockouts} lockouts, {percentage:.1f}% of era total)")

# Circuit-specific analysis
print(f"\n=== CIRCUIT LOCKOUT ANALYSIS ===")
circuit_lockouts = df.groupby('track_clean').agg({
    'nation_clean': ['count', 'nunique'],
    'season': ['min', 'max', 'nunique']
})
circuit_lockouts.columns = ['total_lockouts', 'nations_involved', 'first_lockout', 'last_lockout', 'seasons_with_lockouts']
circuit_lockouts['lockout_rate'] = circuit_lockouts['total_lockouts'] / circuit_lockouts['seasons_with_lockouts']

# Most lockout-prone circuits
lockout_prone = circuit_lockouts.nlargest(10, 'total_lockouts')
print("Most lockout-prone circuits:")
for i, (circuit, data) in enumerate(lockout_prone.iterrows(), 1):
    lockouts = int(data['total_lockouts'])
    nations = int(data['nations_involved'])
    rate = data['lockout_rate']
    print(f"{i:2d}. {circuit}: {lockouts} lockouts ({nations} nations, {rate:.2f}/season)")

# Most competitive circuits (many different nations)
competitive_circuits = circuit_lockouts.nlargest(10, 'nations_involved')
print(f"\nMost competitive circuits (diverse lockouts):")
for i, (circuit, data) in enumerate(competitive_circuits.iterrows(), 1):
    nations = int(data['nations_involved'])
    lockouts = int(data['total_lockouts'])
    diversity = nations / lockouts if lockouts > 0 else 0
    print(f"{i:2d}. {circuit}: {nations} different nations ({lockouts} total, {diversity:.2f} diversity)")

# Temporal trends
print(f"\n=== TEMPORAL TRENDS ===")
decade_trends = df.groupby('decade').agg({
    'nation_clean': ['count', 'nunique']
})
decade_trends.columns = ['total_lockouts', 'nations_involved']
decade_trends['lockouts_per_nation'] = decade_trends['total_lockouts'] / decade_trends['nations_involved']

print("Lockout trends by decade:")
print(decade_trends.round(2))

# Visualization: Temporal evolution
plt.figure(figsize=(14, 8))
decade_data = df.groupby(['decade', 'nation_clean']).size().unstack(fill_value=0)
top_nations = nation_metrics.head(5).index

for nation in top_nations:
    if nation in decade_data.columns:
        plt.plot(decade_data.index, decade_data[nation], marker='o', label=nation, linewidth=2)

plt.xlabel('Decade')
plt.ylabel('Number of Lockouts')
plt.title('Evolution of Podium Lockouts by Top 5 Nations')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Statistical Modeling and Predictions

In [ ]:
print("=== STATISTICAL MODELING AND PREDICTIONS ===")

# Lockout probability modeling
total_possible_lockouts = len(df['season'].unique()) * len(df['track_clean'].unique()) * len(df['class_clean'].unique())
actual_lockouts = len(df)
overall_lockout_rate = actual_lockouts / total_possible_lockouts if total_possible_lockouts > 0 else 0

print(f"Overall lockout statistics:")
print(f"  Total recorded lockouts: {actual_lockouts:,}")
print(f"  Estimated possible opportunities: {total_possible_lockouts:,}")
print(f"  Overall lockout probability: {overall_lockout_rate:.4f} ({overall_lockout_rate*100:.2f}%)")

# Nation-specific lockout rates
nation_rates = nation_metrics.copy()
nation_rates['lockout_probability'] = nation_rates['total_lockouts'] / actual_lockouts
nation_rates['efficiency_score'] = nation_rates['dominance_index'] / nation_rates['seasons_active']

print(f"\nTop 5 nations by lockout probability:")
top_prob = nation_rates.nlargest(5, 'lockout_probability')
for i, (nation, data) in enumerate(top_prob.iterrows(), 1):
    prob = data['lockout_probability'] * 100
    lockouts = int(data['total_lockouts'])
    efficiency = data['efficiency_score']
    print(f"{i}. {nation}: {prob:.1f}% probability ({lockouts} lockouts, {efficiency:.1f} efficiency)")

# Circuit lockout prediction
if len(circuit_lockouts) >= 5:
    print(f"\n=== CIRCUIT LOCKOUT PREDICTION MODEL ===")
    
    # Prepare features
    X = circuit_lockouts[['nations_involved', 'seasons_with_lockouts']].fillna(0)
    y = circuit_lockouts['total_lockouts']
    
    # Simple correlation analysis
    corr_nations = stats.pearsonr(X['nations_involved'], y)
    corr_seasons = stats.pearsonr(X['seasons_with_lockouts'], y)
    
    print(f"Circuit lockout correlations:")
    print(f"  Nations involved: r={corr_nations[0]:.3f}, p={corr_nations[1]:.4f}")
    print(f"  Seasons active: r={corr_seasons[0]:.3f}, p={corr_seasons[1]:.4f}")
    
    # Simple prediction model (linear)
    from sklearn.linear_model import LinearRegression
    from sklearn.model_selection import cross_val_score
    
    if len(X) >= 10:  # Minimum sample size
        model = LinearRegression()
        scores = cross_val_score(model, X, y, cv=min(5, len(X)))
        
        print(f"\nPrediction model performance:")
        print(f"  Cross-validation R²: {scores.mean():.3f} (±{scores.std()*2:.3f})")
        
        # Fit full model
        model.fit(X, y)
        
        print(f"\nModel coefficients:")
        print(f"  Nations involved: {model.coef_[0]:.3f}")
        print(f"  Seasons active: {model.coef_[1]:.3f}")
        print(f"  Intercept: {model.intercept_:.3f}")

# Clustering nations by lockout patterns
print(f"\n=== NATION CLUSTERING BY LOCKOUT PATTERNS ===")

if len(nation_metrics) >= 5:
    # Prepare clustering features
    cluster_features = ['total_lockouts', 'lockout_span', 'tracks_dominated', 'classes_dominated']
    X_cluster = nation_metrics[cluster_features].fillna(0)
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_cluster)
    
    # K-means clustering
    n_clusters = min(4, len(nation_metrics))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    
    nation_metrics['cluster'] = clusters
    
    print(f"Nation clusters by lockout patterns:")
    for cluster_id in range(n_clusters):
        cluster_nations = nation_metrics[nation_metrics['cluster'] == cluster_id]
        if len(cluster_nations) > 0:
            avg_lockouts = cluster_nations['total_lockouts'].mean()
            avg_tracks = cluster_nations['tracks_dominated'].mean()
            print(f"\nCluster {cluster_id}: {len(cluster_nations)} nations")
            print(f"  Average lockouts: {avg_lockouts:.1f}")
            print(f"  Average tracks dominated: {avg_tracks:.1f}")
            print(f"  Nations: {list(cluster_nations.index)[:5]}")

# Future predictions
print(f"\n=== FUTURE LOCKOUT PREDICTIONS ===")
recent_years = df[df['season'] >= 2015]
recent_rate = len(recent_years) / (2023 - 2015 + 1)  # Approximate recent annual rate
print(f"Recent lockout rate (2015+): {recent_rate:.1f} lockouts per year")

# Identify emerging nations
emerging_nations = df[df['season'] >= 2000]['nation_clean'].value_counts()
established_nations = df[df['season'] < 2000]['nation_clean'].value_counts()

print(f"\nEmerging lockout nations (since 2000):")
for nation in emerging_nations.index[:5]:
    modern_lockouts = emerging_nations[nation]
    historical_lockouts = established_nations.get(nation, 0)
    if modern_lockouts > historical_lockouts:
        print(f"  {nation}: {modern_lockouts} recent vs {historical_lockouts} historical")

print(f"\n✅ PODIUM LOCKOUTS MODELING COMPLETED")
print(f"Key insights: Clear national hierarchies in lockout dominance with")
print(f"circuit-specific patterns and evolving competitive landscapes.")